In [1]:
# aim: sleep staging of icu patients' data. first step is to create hdf5 file.

In [1]:
%matplotlib widget
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import sys
sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import load_sleep_data, write_to_hdf5_file

sys.path.append('/home/wolfgang/repos/AirGoSleepPyT0')
from segment_signal import *
from collections import Counter
from scipy import io as sio

import torch as th
from utils import softmax
from braindecode.torch_ext.util import np_to_var, var_to_np

In [2]:
def clip_z_normalize(signal):

    signal_clipped = np.clip(signal.dropna(), np.percentile(signal.dropna(),1), np.percentile(signal.dropna(),99))
    mean = np.mean(signal_clipped.astype(np.float32))
    std = np.std(signal_clipped.astype(np.float32))
    signal = (signal - mean)/std

    return signal

 
def myprint(seg_mask):
    sm = Counter([x.split('_')[0] for x in seg_mask])
    for ex in seg_mask_explanation:
        if ex in sm:
            print(('%s: %d/%d, %g%%'%(ex,sm[ex],len(seg_mask),sm[ex]*100./len(seg_mask))))
    

In [4]:
def segment_signal_only(signal, window_time, step_time, fs, 
                        notch_freq=None, bandpass_freq=None, start_end_remove_window_num=1, 
                        amplitude_thres=None, amplitude_thres_low=None, n_jobs=-1, 
                        to_remove_mean=False):

    # segmentation:
    std_thres1 = 0.00001 
    std_thres2 = 0.00001

    window_size = int(round(window_time*fs))
    step_size = int(round(step_time*fs))

    # prepare segmentation:
    start_ids = np.arange(0, signal.shape[1]-window_size+1, step_size)
    if start_end_remove_window_num>0:
        start_ids = start_ids[start_end_remove_window_num:-start_end_remove_window_num]
    seg_masks = [seg_mask_explanation[0]]*len(start_ids)

    ### check for high amplitude (airgo not worn) before detrending signal:
    if amplitude_thres is not None:
    ## find large amplitude in signal
        signal_segs_temp = signal[:,[np.arange(x,x+window_size) for x in start_ids]].transpose(1,0,2)  # (#window, #ch, window_size+2padding)

        amplitude_large2d = np.any(np.abs(signal_segs_temp)>amplitude_thres, axis=2)
        amplitude_large1d = np.where(np.any(amplitude_large2d, axis=1))[0]
        for i in amplitude_large1d:
            seg_masks[i] = '%s'%(seg_mask_explanation[4])

    if amplitude_thres_low is not None:
        ## find low amplitude in signal
        signal_segs_temp = signal[:,[np.arange(x,x+window_size) for x in start_ids]].transpose(1,0,2)  # (#window, #ch, window_size+2padding)
        amplitude_low2d = np.any(np.abs(signal_segs_temp)<amplitude_thres_low, axis=2)
        amplitude_low1d = np.where(np.any(amplitude_low2d, axis=1))[0]
        for i in amplitude_low1d:
            seg_masks[i] = '%s'%(seg_mask_explanation[4])

    # normalize signal:
    signal = clip_z_normalize(pd.Series(signal[0,:]))
    signal = signal[np.newaxis, :]

    # create signal segments:
    signal_segs = signal[:,[np.arange(x,x+window_size) for x in start_ids]].transpose(1,0,2)  # (#window, #ch, window_size+2padding)

    ## find nan in signal segs

    nan2d = np.any(np.isnan(signal_segs), axis=2)
    nan1d = np.where(np.any(nan2d, axis=1))[0]
    for i in nan1d:
        seg_masks[i] = '%s'%(seg_mask_explanation[3])

    return signal_segs, seg_masks, start_ids

In [5]:

seg_mask_explanation = [
    'normal',
    'around sleep stage change point',
    'NaN in sleep stage',
    'NaN in signal',
    'overly high/low amplitude',
    'flat signal',
    'NaN in feature',
    'NaN in spectrum',
    'muscle artifact',
    'spurious spectrum',
    'missing or additional R peak']


In [6]:
# step 1: load data

In [7]:
input_dir = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/biosignals_10Hz_data'
output_dir = '/media/ssd2/icu_sleep_sleepstaging/h5'

verbose = True
files = os.listdir(input_dir)
files.sort()
print(files[:3])

window_time = 270
step_time =30
start_end_remove_window_num=1
n_jobs=-1
to_remove_mean=False


['icusleep_001.h5', 'icusleep_002.h5', 'icusleep_003.h5']


In [8]:
models_dir = '/media/mad3/Projects/AirGoSleepStaging/final_models'

cnn_model = os.path.join(models_dir, 'CNN_AirGoFilter_AmendSumEffort.pth')
cnn_model = th.load(cnn_model);
cnn_model.eval();

lstm_model = os.path.join(models_dir, 'RNN_AirGoFilter_AmendSumEffort.pth')
lstm_model = th.load(lstm_model);
lstm_model.eval();

In [9]:
def print_data_availability(series):
    
        print(f'shape total: {series.shape}')
        print(f'shape w/o NA: {series.dropna().shape}')
        print(f'# NA: {pd.isna(series).sum()}')

        print('in hours:')
        print(f'shape total: {series.shape[0]/fs/3600 :.1f}')
        print(f'shape w/o NA: {series.dropna().shape[0]/fs/3600:.1f}')
        print(f'# NA: {pd.isna(series).sum()/fs/3600:.1f}')


In [10]:
def cnn_and_lstm_predict(X, cnn_model, lstm_model):

    output, H = cnn_model(X)
    output = var_to_np(output)
    H = var_to_np(H)

    H = np.expand_dims(H, axis=0)
    H = np_to_var(H)
    H = H.cuda()

    yp = lstm_model(H)[0]
    yp = var_to_np(yp)
    yp = np.squeeze(yp)

    yp = softmax(yp)

    return yp


In [11]:
for file_tmp in files[:1]:
    
    feature_path = os.path.join(output_dir, file_tmp.replace('h5','mat'))
    data, hdr = load_sleep_data(os.path.join(input_dir, file_tmp), idx_to_datetime=1)
    data[np.isinf(data)] = np.nan
    data = data.astype('float64')         
    fs = hdr['fs']
    data.columns = [x.lower() for x in data.columns]

    # print(data.band.dropna().shape[0])
    if data.band.dropna().shape[0] == 0:
        print('No AirGo data, seems like wrong category assigned')
    #     continue # todo

    data.rename({'spo2%':'spo2'}, axis=1, inplace=True)
    #we need different airgo features here for apnea detection. don't use the old computed features here but compute only the features needed for the prediction. keep bedmaster data
    features_to_keep = ['art1d', 'art1s', 'hr', 'spo2', 'accx', 'accy', 'accz', 'band']
    features_to_keep = [x for x in features_to_keep if x in data.columns]

    data = data[features_to_keep]

    data['band_smooth'] = data['band'].reset_index()['band'].rolling(int(0.5*fs), center=True, min_periods=1).mean().values
    
    if verbose:
        print_data_availability(data.band_smooth)

    signal = data.band_smooth.values
    signal = signal[np.newaxis, :]
    
    amplitude_thres = 4075
    amplitude_thres_low = 400 # only in use for AirGo.

    if any(pd.Series(signal[0,:]).dropna()>5000):
        print('airgo version pre V5, remove amplitude thresholds')
        amplitude_thres = 100000000
        amplitude_thres_low = 0 # only in use for AirGo.

    segs, seg_mask, seg_start_pos = segment_signal_only(signal, window_time, step_time, fs, 
                            start_end_remove_window_num=start_end_remove_window_num, 
                            amplitude_thres=amplitude_thres, amplitude_thres_low=amplitude_thres_low,
                        n_jobs=n_jobs, to_remove_mean=to_remove_mean)
    
    normal_only = True
    if normal_only:
        good_ids = np.where(np.in1d(seg_mask,seg_mask_explanation[:2]))[0]
        if len(good_ids)<=0:
            myprint(seg_mask)
            raise ValueError('No normal segments')
        segs = segs[good_ids]
        seg_start_pos = seg_start_pos[good_ids]
    else:
        good_ids = np.arange(len(seg_mask))

    # REMOVE INF and NAN if still here...
    # segs.shape
    isgood = np.isfinite(segs).all(axis=2).flatten()
    segs = segs[isgood,:,:]
    seg_start_pos = seg_start_pos[isgood]

    X = th.tensor(segs).float().cuda()
    
    yp = cnn_and_lstm_predict(X, cnn_model, lstm_model)
    
    yp = np.argmax(yp, axis=1) + 1

    data['stage'] = np.nan
    stage_loc = data.iloc[seg_start_pos].index
    data.loc[stage_loc, 'stage'] = yp
    data['stage'].fillna(method='pad',limit=30*fs-1, inplace=True)
    
    
#     myprint(seg_mask)
    
#     start_datetime = np.array([hdr['start_datetime'].year,hdr['start_datetime'].month,hdr['start_datetime'].day, hdr['start_datetime'].hour, hdr['start_datetime'].minute, hdr['start_datetime'].second, hdr['start_datetime'].microsecond])
    
#     sio.savemat(feature_path, 
#     {
#     'segs': segs,
#     'seg_start_pos': seg_start_pos,
#     'seg_mask': seg_mask,
#     'Fs': fs,
#     'study_id': hdr['study_id'],
#     'start_datetime': start_datetime,
#     }, 
#     do_compression=True)


shape total: (2853171,)
shape w/o NA: (469591,)
# NA: 2383580
in hours:
shape total: 79.3
shape w/o NA: 13.0
# NA: 66.2
airgo version pre V5, remove amplitude thresholds


/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:24: RuntimeWarning: invalid value encountered in greater
/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:32: RuntimeWarning: invalid value encountered in less


In [12]:
hdr['study_id'] = np.int32(hdr['study_id'])
hdr['MRN'] = np.int32(hdr['MRN'])
hdr['fs'] = np.int32(hdr['fs'])
hdr['start_datetime'] = np.array([hdr['start_datetime'].year,hdr['start_datetime'].month,hdr['start_datetime'].day, hdr['start_datetime'].hour, hdr['start_datetime'].minute, hdr['start_datetime'].second, hdr['start_datetime'].microsecond])



In [14]:
write_to_hdf5_file(data, os.path.join(output_dir, file_tmp), hdr=hdr)

In [ ]:
write_to_hdf5_file?

In [ ]:
plt.figure()
plt.plot(data.sleep_stage)
plt.show()

In [ ]:
data.head()

In [ ]:
# # reconstructing datetime from seg_pos:

f = sio.loadmat(feature_path)

import datetime

print(hdr['start_datetime'] + datetime.timedelta(seconds=seg_start_pos[100]/fs))
print(data.iloc[[seg_start_pos[100]]].index)

In [ ]:
write_to_hdf5_file?